In [2]:
from pyspark.sql import functions as F

transaction_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/pyspark_optimization/data/data_skew/transactions.parquet/"
customer_path =  "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/pyspark_optimization/data/data_skew/customers.parquet/"

df_transactions = spark.read.format('parquet').load(transaction_path)
df_customers = spark.read.format('parquet').load(customer_path)

In [3]:
df_transactions.rdd.getNumPartitions()

In [4]:
display(df_transactions.limit(5))

## Narrow Transformation



In [6]:
df_narrow_transform = (
    df_customers
    .filter(F.col("city") == "boston")
    .withColumn("first_name", F.split("name", " ").getItem(0))
    .withColumn("last_name", F.split("name", " ").getItem(1))
    .withColumn("age", F.col("age") + F.lit(5))
    .select("cust_id", "first_name", "last_name", "age", "gender", "birthday")
)
## is noop - is no operation it will no execute but write down the final physical plan
df_narrow_transform.write.format("noop").mode("overwrite").save("../data/test/df_narrow_transform.parquet")

## Wide Transformation

In [10]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 1)

In [13]:
df_joined = (
    df_transactions.join(
        F.broadcast(df_customers),
        how="inner",
        on="cust_id"
    )
)

In [14]:
df_joined.write.format("noop").mode("overwrite").save("../data/test/df_joined.parquet")